In [1]:
import pickle
import ast
import json
import time
import requests
from datetime import datetime
from tqdm import tqdm
import pandas as pd
import numpy as np
import spacy

In [2]:
pd.set_option('display.max_columns', 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [4]:
import os
import s3fs

# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

In [3]:
presse = pd.read_excel("/Users/pol/Downloads/press_corpus.xlsx")

# General NER retrieval

In [4]:
nlp = spacy.load('fr_core_news_lg')

In [78]:
df

,id,source,article_text
0,1,telerama,Serge Reggiani - Succès & confidences .\n Une ...
1,2,telerama,"Désolés, Warren .\n C'est une maigre consolati..."
2,3,telerama,Adam Green - Friends of mine .\n Dire qu'on a ...
3,4,telerama,Cesaria Evora - Voz d'amor .\n Dès les premièr...
4,5,telerama,Herman Düne - Mas cambios et Mash Concrete Met...
...,...,...,...
52689,52690,lemonde,"Rap, jazz, rock, classique, chanson… Les album..."
52690,52691,lemonde,Que sait-on vraiment des surrisques de contami...
52691,52692,lemonde,"Rencontre avec Riccardo Muti, maestro à vie .\..."
52692,52693,lemonde,Ils sont passés chez Colors : la sélection mus...


In [94]:
df = presse

docs = nlp.pipe(
    zip(df["article_text"], df["id"]),
    as_tuples=True,
    disable=['tok2vec', 'parser', 'lemmatizer'],
    batch_size=10,
    n_process=4
)

result = []

with tqdm(total=len(df)) as pbar:
    for doc, idx in docs: 
        for ent in doc.ents:
            if  ent.label_ != 'LOC':
                result.append((idx, ent.text, ent.label_))
            
        pbar.update(1)
        
ner_df = pd.DataFrame(result, columns=['id', 'name', 'ner_type'])
        

100%|██████████| 52694/52694 [26:33<00:00, 33.08it/s] 


In [95]:
ner_df["ner_type"].unique()

<ArrowStringArray>
['PER', 'ORG', 'MISC']
Length: 3, dtype: str

In [96]:
ner_df["name_count"] = ner_df["name"].map(ner_df["name"].value_counts())
ner_df

,id,name,ner_type,name_count
0,1,Serge Reggiani - Succès & confidences,PER,1
1,1,Reggiani,ORG,36
2,1,Brel,PER,337
3,1,Barbara,PER,663
4,1,Ventura,PER,16
...,...,...,...,...
1472688,52694,Moussorgsky,PER,4
1472689,52694,Galina Vichnevskaïa,PER,10
1472690,52694,Arturo Benedetti Michelangeli,PER,9
1472691,52694,Ballade no 1,MISC,2


In [97]:
extracted_ents = (
    ner_df
    .groupby("name")
    .agg({"id": "first",
          "name_count": "first"})
    .reset_index()
)

extracted_ents.to_csv("/Users/pol/Downloads/extracted_ents_ALL.csv", 
                      sep=";",
                      encoding="utf-8-sig")

In [98]:
article_text = presse[["id", "article_text"]]

agg_matches = (
    ner_df
    .groupby("id").agg({
            "name": list,
        })
    .join(article_text, on="id")
    #.reset_index()
    
)

agg_matches = agg_matches[["article_text", "name", "name_count"]]

agg_matches.to_csv("/Users/pol/Downloads/agg_matches_ALL.csv", 
                   sep=";",
                   encoding="utf-8-sig")
agg_matches

KeyError: "['name_count'] not in index"

In [63]:
agg_matches10k = agg_matches[:10000]

agg_matches10k.to_csv("/Users/pol/Downloads/agg_matches10k.csv", 
                   sep=";",
                   encoding="utf-8-sig")